# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)
# TODO: assert the row count and the quantity total
assert len(lines) == 4, f"expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"expected qty total 7, got {lines['qty'].sum()}"

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

assert len(joined) == 4
assert joined['qty'].sum() == 7

unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched['vendor_id'].unique().tolist())   # ['V-07']
print(unmatched['qty'].sum())                     # 3

joined

['V-07']
3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
# TODO
# Unmatched vendor gets an explicit "Unknown" label rather than being dropped.
# groupby() silently discards NaN keys by default, which would make the 3 Hot Dog
# units vanish and the table total 4 instead of 7. Keeping them visible under a
# clearly-labeled row means the totals reconcile with Q1 and the data problem
# (V-07 missing from the vendor list) stays in front of whoever reads the table.
by_vendor = (
    joined
    .assign(name=joined['name'].fillna('Unknown (V-07 not in vendor list)'))
    .groupby('name', as_index=False)['qty']
    .sum()
    .sort_values('qty', ascending=False)
    .reset_index(drop=True)
)

assert by_vendor['qty'].sum() == 7

by_vendor

,name,qty
0,Hoos Burgers,3
1,Unknown (V-07 not in vendor list),3
2,Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [7]:
# TODO
by_zone = (
    joined
    .assign(zone=joined['zone'].fillna('Unknown'))
    .groupby('zone', as_index=False)['qty']
    .sum()
    .sort_values('qty', ascending=False)
    .reset_index(drop=True)
)

assert by_zone['qty'].sum() == 7

# Rank only the zones we actually know about
known = by_zone[by_zone['zone'] != 'Unknown']
print(known.iloc[0].tolist())   # ['A', 4]

by_zone
# Zone A: Hoos Burgers (3) + Cav Merch (1) = 4 units.
# V-07 has no zone, so its 3 units have to be kept visible, or the zone totals
# add up to 4 instead of 7. Fill NaN with 'Unknown' so they reconcile.

['A', np.int64(4)]


,zone,qty
0,A,4
1,Unknown,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [8]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

# json_normalize(record_path='lines') raises KeyError on order 5, because
# errors='ignore' only covers meta keys, not the record_path. So give every
# order a lines list first. A missing or empty list becomes one empty
# placeholder record, so the order still shows up as a row instead of vanishing.
safe = [{**o, 'lines': o.get('lines') or [{}]} for o in ragged_json]

rag = (
    pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])
    .reindex(columns=['order', 'vendor_id', 'item', 'qty'])   # guarantees columns exist
)

# Nullable integer: 0 stays 0, a missing qty becomes <NA>. No fillna(0).
rag['qty'] = rag['qty'].astype('Int64')

# Say why each row looks the way it does
rag['line_status'] = 'ok'
rag.loc[rag['item'].isna(), 'line_status'] = 'no_lines'
rag.loc[rag['item'].notna() & rag['qty'].isna(), 'line_status'] = 'qty_missing'

rag

,order,vendor_id,item,qty,line_status
0,4,V-01,Soda,2,ok
1,5,V-10,NaN,<NA>,no_lines
2,6,V-01,Fries,<NA>,qty_missing


### Q6 — Validate

In [9]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) Why one row per line item

Line items are the finest grain in the data, and the grain you pick decides which questions are cheap. Every question here is a groupby over a column of line items: units per vendor, units per zone, units per item. With one row per line, each of those is a single groupby(...)['qty'].sum().

With one row per order, the line items sit in a list inside a cell. To ask "how many Cheeseburgers sold?" you would have to loop over every list, pull out the matching dicts and add them up by hand, and you would be doing that on every question.

Orders also don't fit one vendor or one zone in a clean way. In this data each order happens to have a single vendor, but an order can hold several items and several quantities, so any per-order cell for item or qty has to be a list or a pre-summed number. A pre-summed number has already thrown away the per-item detail, and you can't get it back.

Going finer to coarser is always possible: group lines by order and you recover per-order totals. Going coarser to finer is not. The line-item shape keeps every question answerable.

It also made the join safe. vendor_id lives on the order but gets copied onto each line by meta, so a plain merge works and the quantity total is preserved (7 before and after).

b) Missing quantity vs. zero

They mean different things. Zero is a fact: the line exists and nothing was sold or ordered. Missing is an absence of knowledge: the line exists but we don't know how much. One is a measurement, the other is a gap in the data.

Conflating them, usually via fillna(0), breaks things quietly:

Totals look complete when they aren't. In Q5 the true sum is "at least 2, with one unknown." After fillna(0) it reads as exactly 2, and nobody is told the figure is a lower bound.
Averages shift. The mean quantity per line over [2, <NA>] is 2 if you skip the gap, but 1 if you treat the gap as zero. Same data, different answer, and the second one is wrong.
Zero-based logic misfires. Filters like qty == 0 (cancelled or empty lines) would flag order 6's Fries as a zero-quantity line when the real situation is that the data feed dropped a field. That sends the problem to the wrong team: someone investigates a cancellation instead of the upstream payload.
The data-quality signal disappears. A missing field points to a producer-side bug or an incomplete record. Once it is overwritten with 0 you can no longer count how often it happens or trace it to a source.

Keeping <NA> (with Int64) and a line_status column preserves all of that. The downstream code then has to decide on purpose what to do with gaps: exclude them, flag them, or impute them with a documented rule. That decision should be explicit, and fillna(0) makes it silently.